# Testing with Dask

Tyumen Arayk for model testing

In [2]:
# setup imports
import sys
from pathlib import Path
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
PROJECT_ROOT = Path().resolve().parents[2]  # pas aan als notebook dieper/dichter zit
sys.path.append(str(PROJECT_ROOT))

In [25]:
from src.models import *
from src.forcing import load_lumped_forcing_data,generate_lumped_ERA5_forcing
from src.paths import *
from ewatercycle.observation.grdc import get_grdc_data

from scipy.optimize import minimize
import numpy as np

import cma
import seaborn as sns

from dask import delayed, compute
from dask.distributed import Client, progress

import warnings
warnings.filterwarnings("ignore", category=UserWarning)

In [27]:
shape_name="Tyumen_Aryk_GRDC"
start_date = '1953-01-02T00:00Z'
end_date = '1956-12-31T00:00Z'


forcing = generate_lumped_ERA5_forcing(shape_name, start=start_date,end=end_date)





In [28]:
ERA5_forcing_loaded = load_lumped_forcing_data("Tyumen_Aryk_GRDC", "ERA5", "1953-1956")


grdc_station = get_grdc_data(2617110,
                   '1953-01-02T00:00Z',
                   '1956-12-31T00:00Z',
                   data_home=GRDC/'Daily')
                   
q_obs = grdc_station['streamflow']
years = q_obs['time'].dt.year.values


In [29]:
# CMA-ES test setup
x0_norm = scale(0.5 * (p_min + p_max))  # start in the middle
sigma0 = 0.1  # step size in normalized space

# Define 20 unique seeds
seeds = [
    180988, 214025, 987654, 123456, 654321, 111222, 333444, 555666,
    777888, 999000, 112233, 445566, 778899, 101010, 202020, 303030,
    404040, 505050, 606060, 707070
]

# results = []

# for s in seeds:
#     res = run_cma(
#         seed=s,
#         x0_norm=x0_norm,
#         sigma0=sigma0,
#         objective_fn=objective_safe,
#         popsize=18,
#         maxfevals=360,
#         save_path=f"results/result_seed_{s}.pkl",
#     )
#     results.append(res)

In [30]:
from dask.distributed import Client

# Create a Dask client with 3 workers, 1 thread each, dashboard auto-assigned
client = Client(n_workers=3, threads_per_worker=1, dashboard_address=":0")

# Display client info (clickable dashboard link appears in Jupyter)
client


Connection method: Cluster object,Cluster type: distributed.LocalCluster
Dashboard: http://127.0.0.1:45565/status,
Dashboard: http://127.0.0.1:45565/status,Workers: 3
Total threads: 3,Total memory: 31.34 GiB
Status: running,Using processes: True
Comm: tcp://127.0.0.1:35471,Workers: 3
Dashboard: http://127.0.0.1:45565/status,Total threads: 3
Started: Just now,Total memory: 31.34 GiB
Comm: tcp://127.0.0.1:45971,Total threads: 1
Dashboard: http://127.0.0.1:40013/status,Memory: 10.45 GiB
Nanny: tcp://127.0.0.1:45505,


In [ ]:
# client = Client()
# futures = client.futures  # all submitted tasks

# # Print active, pending, finished
# for f in futures:
#     print(f.key, f.status)


In [ ]:
# from dask.distributed import Client

# # Assuming you created a client earlier
# client.close()  # closes client and shuts down local cluster/workers


In [ ]:
# from dask.distributed import Client, default_client
# import warnings

# def close_dask():
#     try:
#         # Try to get the current default client
#         client = default_client()
#         print(f"Closing existing Dask client with {len(client.scheduler_info()['workers'])} workers...")
#         client.close()
#         print("Dask client closed successfully.")
#     except ValueError:
#         # No active client
#         print("No active Dask client found.")
#     except Exception as e:
#         warnings.warn(f"Error closing Dask client: {e}")

# # Run the cleanup
# close_dask()

Closing existing Dask client with 3 workers...
Dask client closed successfully.


In [8]:
# from dask import delayed, compute

# objective_fn = make_objective_safe(ERA5_forcing_loaded, q_obs.values, shape_name)
# number_runs = 10
# test_seeds = seeds[:number_runs]

# tasks = [delayed(run_cma)(
#     seed=s,
#     x0_norm=x0_norm,
#     sigma0=sigma0,
#     objective_fn=objective_fn,
#     popsize=10,
#     maxfevals=40,
#     save_folder="results_2/"
# ) for s in test_seeds]

# results = compute(*tasks)
# #print(results)

In [31]:
from dask import delayed
from dask.distributed import progress

objective_fn = make_objective_safe(ERA5_forcing_loaded, q_obs.values, shape_name)
number_runs = 12
test_seeds = seeds[:number_runs]

tasks = [
    delayed(run_cma)(
        cma_seed=s,
        x0_norm=x0_norm,
        sigma0=sigma0,
        objective_fn=objective_fn,
        popsize=16,
        maxfevals=330,
        save_folder="results_5/"
    ) for s in test_seeds
]

# Submit tasks to the cluster
futures = client.compute(tasks)

# Optional: see a live progress bar
progress(futures)

# Gather results when done
results = client.gather(futures)


/opt/conda/envs/ewatercycle2/lib/python3.12/site-packages/esmvalcore/experimental/_warnings.py:13: UserWarning: 
  Thank you for trying out the new ESMValCore API.
  Note that this API is experimental and may be subject to change.
  More info: https://github.com/ESMValGroup/ESMValCore/issues/498
/opt/conda/envs/ewatercycle2/lib/python3.12/site-packages/esmvalcore/experimental/_warnings.py:13: UserWarning: 
  Thank you for trying out the new ESMValCore API.
  Note that this API is experimental and may be subject to change.
  More info: https://github.com/ESMValGroup/ESMValCore/issues/498
/opt/conda/envs/ewatercycle2/lib/python3.12/site-packages/esmvalcore/experimental/_warnings.py:13: UserWarning: 
  Thank you for trying out the new ESMValCore API.
  Note that this API is experimental and may be subject to change.
  More info: https://github.com/ESMValGroup/ESMValCore/issues/498


(8_w,16)-aCMA-ES (mu_w=4.8,w_1=32%) in dimension 9 (seed=445566, Thu Jan 29 10:10:50 2026)
(8_w,16)-aCMA-ES (mu_w=4.8,w_1=32%) in dimension 9 (seed=654321, Thu Jan 29 10:10:50 2026)
(8_w,16)-aCMA-ES (mu_w=4.8,w_1=32%) in dimension 9 (seed=111222, Thu Jan 29 10:10:50 2026)
Iterat #Fevals   function value  axis ratio  sigma  min&max std  t[m:s]
    1     16 7.392504619520340e-01 1.0e+00 8.57e-02  8e-02  9e-02 1:52.2
Iterat #Fevals   function value  axis ratio  sigma  min&max std  t[m:s]
    1     16 7.392504619520340e-01 1.0e+00 8.63e-02  8e-02  9e-02 1:52.9
Iterat #Fevals   function value  axis ratio  sigma  min&max std  t[m:s]
    1     16 5.819255554071069e-01 1.0e+00 1.06e-01  1e-01  1e-01 1:55.3
    2     32 7.137149639217801e-01 1.2e+00 8.29e-02  8e-02  9e-02 3:42.8
    2     32 5.925845902793034e-01 1.4e+00 1.15e-01  1e-01  1e-01 3:44.8
    2     32 6.773306452442290e-01 1.2e+00 8.92e-02  8e-02  9e-02 3:47.1
    3     48 4.924314800132101e-01 1.3e+00 9.56e-02  9e-02  1e-01 5:33.8


In [32]:
# Convert results (list of dicts) to a nice table
df_results = pd.DataFrame(results)

# Optional: reorder columns
df_results = df_results[["seed", "best_f", "best_x", "nfev"]]

# Display nicely
df_results

,seed,best_f,best_x,nfev
0,180988,0.180764,"[0.32231456531171443, 0.40078593361810955, 0.2...",336
1,214025,0.177491,"[0.5979921742225747, 0.6903929362795285, 0.179...",336
2,987654,0.193121,"[0.4830226153769869, 0.7401549755980262, 0.094...",336
3,123456,0.174590,"[0.502046021414256, 0.8364050082974221, 0.2035...",336
4,654321,0.171918,"[0.12285860259111689, 0.2733450795453007, 0.46...",336
5,111222,0.182666,"[0.848411772957325, 0.8910600000783621, 0.2650...",336
6,333444,0.175142,"[0.516312083914876, 0.9188921676553622, 0.1393...",336
7,555666,0.170937,"[0.8983756900396525, 0.613168312881351, 0.1948...",336
8,777888,0.179092,"[0.4679885512802951, 0.37324866653289307, 0.18...",336
9,999000,0.196031,"[0.332382939516388, 0.4788774114851737, 0.1569...",336


In [34]:
import pickle
from pathlib import Path

results_folder = Path("results_5")
result_files = list(results_folder.glob("result_seed_*.pkl"))

loaded_results = []
for f in result_files:
    with open(f, "rb") as file:
        res = pickle.load(file)
        loaded_results.append(res)

# Example: print best objective per seed
for r in loaded_results:
    print(f"Seed {r['seed']}: best_f = {r['best_f']}, nfev = {r['nfev']}")


Seed 654321: best_f = 0.17191763464028345, nfev = 336
Seed 445566: best_f = 0.19949317983489395, nfev = 336
Seed 111222: best_f = 0.18266575766731064, nfev = 336
Seed 214025: best_f = 0.1774908250283217, nfev = 336
Seed 112233: best_f = 0.17496212376385142, nfev = 336
Seed 987654: best_f = 0.19312110455840456, nfev = 336
Seed 180988: best_f = 0.18076396819507898, nfev = 336
Seed 555666: best_f = 0.17093738634993197, nfev = 336
Seed 123456: best_f = 0.17459049481278965, nfev = 336
Seed 999000: best_f = 0.19603113310254358, nfev = 336
Seed 777888: best_f = 0.17909213433566995, nfev = 336
Seed 333444: best_f = 0.1751420671288759, nfev = 336


In [ ]:
best_params_list = []

for run_idx, run in enumerate(results):
    seed = run['seed']
    history = run['history']
    
    # find the best evaluation by objective value
    best_idx = np.argmin(history['objective'])
    
    best_theta_phys = history['theta_phys'][best_idx]
    best_obj = history['objective'][best_idx]
    best_nse = history['nse'][best_idx]
    best_kge = history['kge'][best_idx]
    best_vol_err = history['vol_err'][best_idx]
    
    row = {
        'seed': seed,
        'best_eval': best_idx + 1,
        'objective': best_obj,
        'nse': best_nse,
        'kge': best_kge,
        'vol_err': best_vol_err
    }
    
    # add each parameter with its proper name
    for i, th in enumerate(best_theta_phys):
        row[parameter_names[i]] = th
    
    best_params_list.append(row)

# create DataFrame
best_params_df = pd.DataFrame(best_params_list)
best_params_df

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

# Suppose you have multiple seeds in `results`:
dfs = []
for seed_idx, res_seed in enumerate(results):
    hist = res_seed['history']
    df = pd.DataFrame(hist)
    df['seed'] = seed_idx  # add a seed column
    dfs.append(df)

history_df = pd.concat(dfs, ignore_index=True)

n_params = len(p_min)  # 9
seeds = history_df['seed'].unique()  # get all unique seeds
colors = plt.cm.tab10(np.linspace(0, 1, len(seeds)))  # assign colors

n_cols = 3
n_rows = int(np.ceil(n_params / n_cols))

fig, axes = plt.subplots(n_rows, n_cols, figsize=(5*n_cols, 3*n_rows))
axes = axes.flatten()  # flatten for easy indexing

for i in range(n_params):
    ax = axes[i]
    
    for seed, color in zip(seeds, colors):
        theta_array = np.stack(history_df.loc[history_df['seed']==seed, 'theta_phys'].values)
        obj_array = history_df.loc[history_df['seed']==seed, 'objective'].values
        ax.scatter(theta_array[:, i], obj_array, label=f"Seed {seed}", color=color, alpha=0.3, s=10)
        ax.set_xlabel(parameter_names[i])
    
    ax.set_ylabel("Objective")
    #ax.set_title()
    ax.set_xlim(p_min[i], p_max[i])
    ax.grid(True)

# remove empty axes
for j in range(n_params, len(axes)):
    fig.delaxes(axes[j])


#axes[0].legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()


In [ ]:
# Parameter columns
param_cols = ["Imax","Ce","Sumax","Beta","Pmax","Tlag","Kf","Ks","FM"]

# Scale the best parameters
scaled_params_df = pd.DataFrame(scale(best_params_df[param_cols].to_numpy()), 
                                columns=param_cols)


plt.figure(figsize=(12,4))
# sns.boxplot(data=scaled_params_df, color = 'lightgray')
# sns.stripplot(data=scaled_params_df, color='black', jitter=False, size=8)
sns.boxplot(data=scaled_params_df, color='lightgray', fliersize=0)  # light gray boxes, hide default outliers
sns.stripplot(data=scaled_params_df, color='black', jitter=True, size=6, marker='o')  # black dots

plt.xticks(rotation=45)
plt.ylabel("Scaled parameter value (0-1)")
plt.title("Scaled best parameter sets across seeds")
# add horizontal grid lines
plt.grid(axis='y', linestyle='--', alpha=0.5)

plt.show()

In [35]:
from dask.distributed import Client

# Assuming you created a client earlier
client.close()  # closes client and shuts down local cluster/workers